# Lab Day 10: Qwen3-VL-4B road damage grounding
Họ tên: **Phạm Văn Kiên** · MSV: **2A202602590**

Kaggle T4: executable E0–E4 implementation. Run cells in order. Metrics are generated only from real model predictions. E5 GRPO is optional and is not executed; its IoU reward has CPU checks.


## 0. Environment (Kaggle T4)
Select GPU T4 (or T4 x2), enable Internet, and attach lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes with Add Input. The model uses GPU 0; two T4 cards do not pool VRAM.

To reuse completed E0/E1/E2 from the previous failed run, create a notebook from its Output or attach that Output as Input. If it lacks prepared data/images and data/splits, attach the original road dataset too. The data cell restores completed experiments after checking model/data compatibility. If it requests the original E1 source Output, attach that earlier Output as well. Unfinished runs start over with AMP recovery.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if Path('/kaggle/working').is_dir():
    project = Path('/kaggle/working/day10')
    if not (project / 'scripts/prepare_data.py').is_file():
        subprocess.run(['git', 'clone',
            'https://github.com/pvksssss/K4-Track-4-Day10-VisionLanguageModel.git',
            str(project)], check=True)
    os.chdir(project)
else:
    project = Path.cwd()
    if not (project / 'scripts/prepare_data.py').is_file():
        raise RuntimeError('Run inside the project folder or on Kaggle.')
print('Project:', project)
subprocess.run(['nvidia-smi', '-L'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
               'requirements-colab.txt', 'pytest', 'matplotlib'], check=True)
# This lab uses bitsandbytes, not torchao. Kaggle's optional torchao 0.10.0
# makes PEFT 0.21.2 fail when adding LoRA to ordinary fp16 vision Linear layers.
if Path('/kaggle/working').is_dir():
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
import importlib
importlib.invalidate_caches()
from peft.import_utils import is_torchao_available
is_torchao_available.cache_clear()
if Path('/kaggle/working').is_dir() and is_torchao_available():
    raise RuntimeError('Unexpected torchao backend is still active; restart the kernel')
# Exercise ordinary Linear dispatch now, before hours of E0/E1/E2 execution.
from peft import LoraConfig, get_peft_model
import torch
probe = get_peft_model(torch.nn.Sequential(torch.nn.Linear(2, 2, dtype=torch.float16)),
    LoraConfig(r=1, lora_alpha=2, target_modules=['0']))
assert any('lora_A' in name for name, _ in probe.named_parameters())
del probe
print('PEFT fp16 Linear adapter dispatch preflight passed.')


In [ ]:
STUDENT_NAME = 'Phạm Văn Kiên'
STUDENT_ID = '2A202602590'
print(f'{STUDENT_NAME} - {STUDENT_ID}')

# Embedded implementation survives cloning the original assignment repo.
Path('kaggle_vlm.py').write_text('"""Student implementation of the Day 10 lab; official scripts stay unchanged.\n\nRun on Kaggle: python kaggle_vlm.py --run-all\nNo GPU libraries are imported until a model or training dataset is used.\n"""\nimport argparse\nimport copy\nimport gc\nimport hashlib\nimport json\nimport math\nimport random\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\n\nMODEL_ID = \'Qwen/Qwen3-VL-4B-Instruct\'\nCLASSES = (\'pothole\', \'crack\', \'manhole\')\nSTUDENT_NAME = \'Phạm Văn Kiên\'\nSTUDENT_ID = \'2A202602590\'\n# Transformers 5 matches skip patterns against the full module path.\nQUANTIZATION_SKIP_MODULES = [\'model.visual\', \'visual\', \'lm_head\']\n\n\ndef read_jsonl(path):\n    return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8\').splitlines() if s.strip()]\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef yolo_to_box(label, yolo, width=640, height=360, coord=\'norm1000\'):\n    """Round boxes; expand tiny positive boxes by one unit, omit zero-area labels."""\n    if label not in CLASSES or coord not in (\'norm1000\', \'pixel\'):\n        raise ValueError(\'Unknown class or coordinate scale\')\n    xc, yc, w, h = yolo\n    if not all(math.isfinite(v) and 0 <= v <= 1 for v in yolo):\n        raise ValueError(\'YOLO values must be finite and within [0, 1]\')\n    if width <= 0 or height <= 0:\n        raise ValueError(\'Image dimensions must be positive\')\n    if w == 0 or h == 0:\n        return None\n    sx, sy = (1000, 1000) if coord == \'norm1000\' else (width, height)\n    box = [round(max(0, min(1, v)) * scale) for v, scale in zip(\n        (xc-w/2, yc-h/2, xc+w/2, yc+h/2), (sx, sy, sx, sy))]\n    for lo, hi, limit in ((0, 2, sx), (1, 3, sy)):\n        if box[hi] <= box[lo]:\n            # Keep ordinary round() results unchanged. At the right/bottom edge,\n            # shift the lower endpoint inward rather than exceeding the scale.\n            box[lo] = min(box[lo], limit - 1)\n            box[hi] = box[lo] + 1\n    return {\'label\': label, \'bbox_2d\': box}\n\n\ndef objects_to_answer(objects, width=640, height=360, coord=\'norm1000\'):\n    boxes = [yolo_to_box(o[\'label\'], o[\'yolo\'], width, height, coord) for o in objects]\n    boxes = [b for b in boxes if b is not None]\n    boxes.sort(key=lambda o: (o[\'bbox_2d\'][1], o[\'bbox_2d\'][0], o[\'label\']))\n    return json.dumps(boxes, separators=(\',\', \':\'))\n\n\ndef audit_annotations(rows, coord=\'norm1000\'):\n    """Report label repairs before loading the GPU model; GT files stay unchanged."""\n    stats = {\'images\': len(rows), \'boxes\': 0, \'expanded_tiny_boxes\': 0,\n             \'zero_area_labels_omitted_from_training_answers\': 0, \'coord\': coord}\n    for row in rows:\n        sx, sy = (1000, 1000) if coord == \'norm1000\' else (row[\'width\'], row[\'height\'])\n        for obj in row[\'objects\']:\n            stats[\'boxes\'] += 1\n            converted = yolo_to_box(obj[\'label\'], obj[\'yolo\'], row[\'width\'], row[\'height\'], coord)\n            if converted is None:\n                stats[\'zero_area_labels_omitted_from_training_answers\'] += 1\n                continue\n            xc, yc, w, h = obj[\'yolo\']\n            rounded = [round(max(0, min(1, v))*s) for v, s in zip(\n                (xc-w/2, yc-h/2, xc+w/2, yc+h/2), (sx, sy, sx, sy))]\n            stats[\'expanded_tiny_boxes\'] += int(rounded[2] <= rounded[0] or rounded[3] <= rounded[1])\n    return stats\n\n\ndef make_prompt(coord=\'norm1000\', width=640, height=360):\n    if coord not in (\'norm1000\', \'pixel\'):\n        raise ValueError(\'Unknown coordinate scale\')\n    scale = (\'integer coordinates on a 0-1000 scale, normalized independently \'\n             \'by image width and height\') if coord == \'norm1000\' else (\n             f\'integer pixel coordinates in the original {width}x{height} image\')\n    return (\'Locate all road damage objects in this image. Use only the labels \'\n            \'pothole, crack, manhole. Return only a JSON list of objects with \'\n            \'keys "label" and "bbox_2d": [x1,y1,x2,y2]. Use \' + scale +\n            \'. Sort boxes from top to bottom, then left to right. \'\n            \'Return [] if no objects are present.\')\n\n\ndef user_message(image, prompt):\n    return {\'role\': \'user\', \'content\': [\n        {\'type\': \'image\', \'image\': image}, {\'type\': \'text\', \'text\': prompt}]}\n\n\ndef config_for(setup, batch_size=1, grad_accum=8):\n    import yaml\n    cfg = yaml.safe_load(Path(f\'configs/{setup}.yaml\').read_text(encoding=\'utf-8\'))\n    cfg.update(coord=\'norm1000\', train_fraction=1.0, inference_batch_size=1,\n               max_new_tokens=256)\n    if setup != \'e0\':\n        # Same effective batch (8 images) for E1-E4; lower T4 memory use.\n        cfg.update(batch_size=batch_size, grad_accum=grad_accum)\n    return cfg\n\n\ndef fingerprint(cfg):\n    meta = json.loads(Path(\'data/splits/meta.json\').read_text(encoding=\'utf-8\'))\n    payload = {\'config\': cfg, \'dataset_version\': meta[\'dataset_version\'],\n               \'prompt\': make_prompt(cfg[\'coord\']),\n               \'implementation_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef seed_everything(seed):\n    import numpy as np\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef load_model(cfg):\n    import torch\n    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable GPU T4 in Kaggle Settings before running model cells\')\n    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n        llm_int8_skip_modules=QUANTIZATION_SKIP_MODULES)\n    model = Qwen3VLForConditionalGeneration.from_pretrained(cfg.get(\'model_id\', MODEL_ID),\n        quantization_config=quant, dtype=torch.float16, device_map={\'\': 0},\n        attn_implementation=\'sdpa\')\n    validate_quantization(model)\n    processor = AutoProcessor.from_pretrained(cfg.get(\'model_id\', MODEL_ID))\n    processor.tokenizer.padding_side = \'left\'\n    return model, processor\n\n\ndef validate_quantization(model):\n    """Fail before inference/training if the vision tower or output head is 4-bit."""\n    protected = [(n, p) for n, p in model.named_parameters()\n        if n.startswith((\'model.visual.\', \'visual.\', \'lm_head.\'))]\n    visual = [(n, p) for n, p in protected if \'visual.\' in n]\n    if not visual:\n        raise RuntimeError(\'Cannot locate vision tower; inspect full model module names\')\n    bad = [n for n, p in protected if not p.is_floating_point() or\n           type(p).__name__ in (\'Params4bit\', \'Int8Params\')]\n    if bad:\n        raise RuntimeError(f\'Vision/head quantization exclusion failed: {bad[:5]}\')\n    quantized_llm = [n for n, p in model.named_parameters()\n        if \'language_model.\' in n and type(p).__name__ == \'Params4bit\']\n    if not quantized_llm:\n        raise RuntimeError(\'LLM has no 4-bit weights; check quantization configuration\')\n    print(\'Quantization verified: vision/head floating point; \'\n          f\'{len(quantized_llm)} LLM weights in 4-bit.\', flush=True)\n\n\ndef apply_setup(model, cfg):\n    import torch\n    from peft import LoraConfig, get_peft_model\n    for p in model.parameters():\n        p.requires_grad_(False)\n    model.enable_input_require_grads()\n    llm = r\'.*language_model\\.layers\\.\\d+\\.(?:self_attn\\.(?:q|k|v|o)_proj|mlp\\.(?:gate|up|down)_proj)\'\n    vit = r\'.*visual\\.blocks\\.\\d+\\.(?:attn\\.(?:qkv|proj)|mlp\\.(?:linear_fc1|linear_fc2))\'\n    pattern = f\'(?:{llm}|{vit})\' if cfg[\'lora\'][\'targets\'] == \'llm+vit\' else llm\n    matched = [n for n, _ in model.named_modules() if re.fullmatch(pattern, n)]\n    expected = 348 if cfg[\'lora\'][\'targets\'] == \'llm+vit\' else 252\n    if len(matched) != expected:\n        raise RuntimeError(f\'Expected {expected} LoRA modules, found {len(matched)}: {matched[:10]}\')\n    model = get_peft_model(model, LoraConfig(r=cfg[\'lora\'][\'r\'],\n        lora_alpha=cfg[\'lora\'][\'alpha\'], lora_dropout=cfg[\'lora\'][\'dropout\'],\n        target_modules=pattern, bias=\'none\', task_type=\'CAUSAL_LM\'))\n    # GradScaler cannot unscale fp16 gradients; LoRA optimizer weights stay fp32.\n    for p in model.parameters():\n        if p.requires_grad:\n            p.data = p.data.to(torch.float32)\n    if cfg[\'train_projector\']:\n        found = 0\n        for n, p in model.named_parameters():\n            if \'visual.merger.\' in n or \'visual.deepstack_merger_list.\' in n:\n                if not p.is_floating_point():\n                    raise RuntimeError(\'Projector is quantized; visual must remain unquantized\')\n                p.data = p.data.to(torch.float32)\n                p.requires_grad_(True)\n                found += p.numel()\n        if found < 100_000_000:\n            raise RuntimeError(f\'Unexpected projector parameter count: {found}\')\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    model.config.use_cache = False\n    model.print_trainable_parameters()\n    return model, matched\n\n\ndef encode_example(processor, entry, coord=\'norm1000\'):\n    import torch\n    from PIL import Image\n    with Image.open(Path(\'data\') / entry[\'image_path\']) as im:\n        image = im.convert(\'RGB\')\n    user = user_message(image, make_prompt(coord, entry[\'width\'], entry[\'height\']))\n    answer = objects_to_answer(entry[\'objects\'], entry[\'width\'], entry[\'height\'], coord)\n    prompt = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,\n        return_dict=True, return_tensors=\'pt\')\n    full = processor.apply_chat_template([user, {\'role\': \'assistant\', \'content\': answer}],\n        tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors=\'pt\')\n    n = prompt[\'input_ids\'].shape[1]\n    if not torch.equal(full[\'input_ids\'][0, :n], prompt[\'input_ids\'][0]):\n        raise RuntimeError(\'Chat prompt is not a prefix of the full conversation\')\n    full[\'labels\'] = full[\'input_ids\'].clone()\n    full[\'labels\'][:, :n] = -100\n    if not (full[\'labels\'] != -100).any():\n        raise RuntimeError(\'No answer tokens remain for loss\')\n    return full\n\n\ndef collate(batch, pad_token_id):\n    import torch\n    from torch.nn.utils.rnn import pad_sequence\n    if any(set(b) != set(batch[0]) for b in batch):\n        raise RuntimeError(\'Processor returned inconsistent batch keys\')\n    out = {}\n    for key in batch[0]:\n        values = [b[key] for b in batch]\n        if key in (\'input_ids\', \'attention_mask\', \'mm_token_type_ids\', \'labels\', \'token_type_ids\'):\n            pad = -100 if key == \'labels\' else pad_token_id if key == \'input_ids\' else 0\n            out[key] = pad_sequence([v.squeeze(0) for v in values], batch_first=True, padding_value=pad)\n        elif key in (\'pixel_values\', \'image_grid_thw\'):\n            out[key] = torch.cat(values, dim=0)\n        else:\n            raise RuntimeError(f\'Unexpected processor field {key}; inspect installed transformers\')\n    return out\n\n\ndef selected_train_rows(cfg):\n    rows = read_jsonl(\'data/splits/train.jsonl\')\n    random.Random(cfg[\'seed\']).shuffle(rows)\n    return rows[:max(1, int(len(rows) * cfg[\'train_fraction\']))]\n\n\ndef save_checkpoint(model, processor, directory, cfg, step, run=None):\n    import torch\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    model.save_pretrained(directory)\n    processor.save_pretrained(directory)\n    extra = {n: p.detach().cpu() for n, p in model.named_parameters()\n             if p.requires_grad and (\'visual.merger.\' in n or \'visual.deepstack_merger_list.\' in n)}\n    if cfg[\'train_projector\'] and not extra:\n        raise RuntimeError(\'Projector checkpoint is empty\')\n    if extra:\n        torch.save(extra, directory / \'projector.pt\')\n    write_json(directory / \'manifest.json\', {\'fingerprint\': fingerprint(cfg), \'config\': cfg, \'step\': step})\n    if run is not None:\n        write_json(directory / \'run.json\', run)\n\n\ndef amp_optimizer_step(optimizer, scaler, params, max_norm=1.):\n    """Clear overflowing gradients, lower the scale, and never update bad weights."""\n    import torch\n    scaler.unscale_(optimizer)\n    # Check BEFORE clipping: clipping an infinite gradient can produce NaNs.\n    finite = all(torch.isfinite(p.grad).all().item() for p in params if p.grad is not None)\n    norm = torch.nn.utils.clip_grad_norm_(params, max_norm) if finite else None\n    if not finite or not torch.isfinite(norm).item():\n        new_scale = scaler.get_scale() * scaler.get_backoff_factor()\n        if not math.isfinite(new_scale) or new_scale <= 0:\n            optimizer.zero_grad(set_to_none=True)\n            raise RuntimeError(\'AMP scale exhausted; inspect model activations or reduce LR\')\n        # update(new_scale=...) resets the unscale state for the next retry. This\n        # also handles aggregate norm overflow when individual gradients are finite.\n        scaler.update(new_scale=new_scale)\n        optimizer.zero_grad(set_to_none=True)\n        return False, None\n    scaler.step(optimizer)\n    scaler.update()\n    optimizer.zero_grad(set_to_none=True)\n    return True, norm.item()\n\n\ndef train(cfg, key):\n    import torch\n    from torch.utils.data import DataLoader\n    output = Path(\'work\') / key\n    final = output / \'final\'\n    signature = fingerprint(cfg)\n    if (final / \'run.json\').exists():\n        run = json.loads((final / \'run.json\').read_text(encoding=\'utf-8\'))\n        if run.get(\'fingerprint\') != signature:\n            raise RuntimeError(f\'{final} belongs to another configuration; use a new run name\')\n        print(\'Reuse completed training:\', key)\n        return run\n    seed_everything(cfg[\'seed\'])\n    rows = selected_train_rows(cfg)\n    annotation_stats = audit_annotations(rows, cfg[\'coord\'])\n    write_json(output / \'annotation_audit.json\', annotation_stats)\n    print(\'Annotation audit:\', annotation_stats, flush=True)\n    model, processor = load_model(cfg)\n    try:\n        model, matched = apply_setup(model, cfg)\n        processor.tokenizer.padding_side = \'right\'\n        class Dataset:\n            def __len__(self): return len(rows)\n            def __getitem__(self, i): return encode_example(processor, rows[i], cfg[\'coord\'])\n        loader = DataLoader(Dataset(), batch_size=cfg[\'batch_size\'], shuffle=True,\n            num_workers=0, generator=torch.Generator().manual_seed(cfg[\'seed\']),\n            collate_fn=lambda b: collate(b, processor.tokenizer.pad_token_id))\n        params = [p for p in model.parameters() if p.requires_grad]\n        optimizer = torch.optim.AdamW(params, lr=cfg[\'lr\'], weight_decay=0)\n        warmup = max(1, round(cfg[\'max_steps\'] * .05))\n        def lr_factor(step):\n            if step < warmup: return (step + 1) / warmup\n            progress = (step - warmup) / max(1, cfg[\'max_steps\'] - warmup)\n            return .5 * (1 + math.cos(math.pi * min(1, progress)))\n        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)\n        scaler = torch.amp.GradScaler(\'cuda\', init_scale=4096.)\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        torch.cuda.reset_peak_memory_stats()\n        torch.cuda.synchronize()\n        started = time.perf_counter()\n        iterator = iter(loader)\n        history = []\n        overflow_events = []\n        write_json(output / \'config.json\', cfg)\n        for step in range(1, cfg[\'max_steps\'] + 1):\n            # Keep one effective batch in CPU RAM; retry these same images on\n            # overflow rather than losing data or counting a skipped optimizer step.\n            micro_batches = []\n            for _ in range(cfg[\'grad_accum\']):\n                try: batch = next(iterator)\n                except StopIteration:\n                    iterator = iter(loader)\n                    batch = next(iterator)\n                micro_batches.append(batch)\n            rng_cpu = torch.get_rng_state()\n            rng_cuda = torch.cuda.get_rng_state(0)\n            for retry in range(9):\n                # Reproduce dropout for the same effective batch on each retry.\n                torch.set_rng_state(rng_cpu)\n                torch.cuda.set_rng_state(rng_cuda, device=0)\n                total_loss = 0.\n                for cpu_batch in micro_batches:\n                    batch = {k: v.to(\'cuda:0\') for k, v in cpu_batch.items()}\n                    with torch.autocast(\'cuda\', dtype=torch.float16):\n                        loss = model(**batch).loss\n                    if not torch.isfinite(loss):\n                        raise RuntimeError(f\'Non-finite loss at step {step}; lower LR and inspect labels\')\n                    total_loss += loss.detach().float().item() / cfg[\'grad_accum\']\n                    scaler.scale(loss / cfg[\'grad_accum\']).backward()\n                updated, grad_norm = amp_optimizer_step(optimizer, scaler, params)\n                if updated: break\n                event = {\'step\': step, \'retry\': retry+1, \'new_scale\': scaler.get_scale()}\n                overflow_events.append(event)\n                print(f\'{key} AMP overflow: {event}; retrying the same batch.\', flush=True)\n                write_json(output / \'overflow.json\', overflow_events)\n            else:\n                raise RuntimeError(f\'Persistent non-finite gradients at step {step} after 9 attempts; \'\n                                   \'lower LR and inspect forward/backward activations\')\n            # Advance the LR schedule only after an actual optimizer update.\n            scheduler.step()\n            history.append({\'step\': step, \'loss\': total_loss, \'lr\': scheduler.get_last_lr()[0],\n                            \'grad_norm\': grad_norm, \'amp_scale\': scaler.get_scale()})\n            if step == 1 or step % 10 == 0:\n                print(f\'{key} step={step}/{cfg["max_steps"]} loss={total_loss:.4f}\', flush=True)\n                write_json(output / \'loss.json\', history)\n            if step % 100 == 0 and step != cfg[\'max_steps\']:\n                # One rolling checkpoint limits disk usage. Interrupted training restarts;\n                # optimizer/RNG are intentionally not presented as resumable state.\n                save_checkpoint(model, processor, output / \'checkpoint\', cfg, step)\n        torch.cuda.synchronize()\n        run = {\'trainable_params\': sum(p.numel() for p in params),\n               \'peak_vram_gb\': torch.cuda.max_memory_allocated() / 1024**3,\n               \'train_minutes\': (time.perf_counter() - started) / 60,\n               \'fingerprint\': signature, \'steps\': cfg[\'max_steps\'], \'train_images\': len(rows),\n               \'matched_modules\': len(matched), \'config\': cfg,\n               \'annotation_audit\': annotation_stats,\n               \'amp_policy\': \'retry_same_batch_v1\', \'gradient_overflows\': len(overflow_events),\n               \'final_amp_scale\': scaler.get_scale(),\n               \'student_name\': STUDENT_NAME, \'student_id\': STUDENT_ID}\n        save_checkpoint(model, processor, final, cfg, cfg[\'max_steps\'], run)\n        write_json(output / \'loss.json\', history)\n        print(json.dumps(run, ensure_ascii=False, indent=2))\n        return run\n    finally:\n        # Release the model even after OOM; each experiment starts from the same base.\n        if \'optimizer\' in locals(): del optimizer\n        if \'scheduler\' in locals(): del scheduler\n        if \'iterator\' in locals(): del iterator\n        if \'params\' in locals(): del params\n        if \'batch\' in locals(): del batch\n        if \'micro_batches\' in locals(): del micro_batches\n        if \'cpu_batch\' in locals(): del cpu_batch\n        if \'loss\' in locals(): del loss\n        if \'loader\' in locals(): del loader\n        del model, processor\n        gc.collect()\n        torch.cuda.empty_cache()\n\n\ndef restore_model(cfg, key):\n    import torch\n    from peft import PeftModel\n    model, processor = load_model(cfg)\n    if key != \'e0\':\n        final = Path(\'work\') / key / \'final\'\n        manifest = json.loads((final / \'manifest.json\').read_text(encoding=\'utf-8\'))\n        if manifest[\'fingerprint\'] != fingerprint(cfg):\n            raise RuntimeError(\'Adapter configuration or dataset version does not match\')\n        model = PeftModel.from_pretrained(model, final, is_trainable=False)\n        if cfg[\'train_projector\']:\n            extra = torch.load(final / \'projector.pt\', map_location=\'cpu\', weights_only=True)\n            expected = {n for n, _ in model.named_parameters()\n                        if \'visual.merger.\' in n or \'visual.deepstack_merger_list.\' in n}\n            if set(extra) != expected:\n                raise RuntimeError(\'Projector checkpoint names differ from loaded model\')\n            with torch.no_grad():\n                for n, p in model.named_parameters():\n                    if n in extra:\n                        p.data = p.data.float()\n                        p.copy_(extra[n].to(p.device))\n    model.eval()\n    model.config.use_cache = True\n    return model, processor\n\n\ndef predict_and_score(cfg, key, split=\'test\', prediction_key=None):\n    import torch\n    from PIL import Image\n    pred_key = prediction_key or key\n    pred_path = Path(\'work\') / f\'{pred_key}_{split}.jsonl\'\n    marker = pred_path.with_suffix(\'.manifest.json\')\n    signature = fingerprint(cfg)\n    expected_marker = {\'fingerprint\': signature, \'split\': split, \'model_run\': key}\n    rows = read_jsonl(f\'data/splits/{split}.jsonl\')\n    if pred_path.exists():\n        if not marker.exists() or json.loads(marker.read_text(encoding=\'utf-8\')) != expected_marker:\n            raise RuntimeError(f\'Stale predictions: {pred_path}; use a different run name\')\n        existing = read_jsonl(pred_path)\n        ids = [r[\'image_id\'] for r in existing]\n        if len(ids) != len(set(ids)) or not set(ids) <= {r[\'image_id\'] for r in rows}:\n            raise RuntimeError(\'Predictions contain duplicate or foreign image IDs\')\n        done = set(ids)\n    else:\n        done = set()\n        write_json(marker, expected_marker)\n    pending = [r for r in rows if r[\'image_id\'] not in done]\n    resource_path = pred_path.with_suffix(\'.resources.json\')\n    if pending:\n        model, processor = restore_model(cfg, key)\n        try:\n            previous = json.loads(resource_path.read_text(encoding=\'utf-8\')) if resource_path.exists() else {}\n            torch.cuda.reset_peak_memory_stats()\n            torch.cuda.synchronize()\n            started = time.perf_counter()\n            bs = cfg[\'inference_batch_size\']\n            with pred_path.open(\'a\', encoding=\'utf-8\') as f, torch.inference_mode():\n                for start in range(0, len(pending), bs):\n                    batch = pending[start:start+bs]\n                    conversations = []\n                    for entry in batch:\n                        with Image.open(Path(\'data\') / entry[\'image_path\']) as im:\n                            image = im.convert(\'RGB\')\n                        conversations.append([user_message(image, make_prompt(cfg[\'coord\'],\n                            entry[\'width\'], entry[\'height\']))])\n                    inputs = processor.apply_chat_template(conversations, tokenize=True,\n                        add_generation_prompt=True, return_dict=True, return_tensors=\'pt\',\n                        processor_kwargs={\'padding\': True})\n                    inputs = {k: v.to(\'cuda:0\') for k, v in inputs.items()}\n                    with torch.autocast(\'cuda\', dtype=torch.float16):\n                        output = model.generate(**inputs, max_new_tokens=cfg[\'max_new_tokens\'], do_sample=False)\n                    raw = processor.batch_decode(output[:, inputs[\'input_ids\'].shape[1]:], skip_special_tokens=True)\n                    for entry, text in zip(batch, raw):\n                        f.write(json.dumps({\'image_id\': entry[\'image_id\'], \'raw\': text}, ensure_ascii=False)+\'\\n\')\n                    f.flush()\n                    write_json(resource_path, {\n                        \'inference_minutes\': previous.get(\'inference_minutes\', 0) +\n                            (time.perf_counter() - started) / 60,\n                        \'inference_peak_vram_gb\': max(previous.get(\'inference_peak_vram_gb\', 0),\n                            torch.cuda.max_memory_allocated() / 1024**3)})\n                    if start % 20 == 0:\n                        print(f\'{pred_key} {split}: {len(done)+start+len(batch)}/{len(rows)}\', flush=True)\n        finally:\n            if \'inputs\' in locals(): del inputs\n            if \'output\' in locals(): del output\n            del model, processor\n            gc.collect()\n            torch.cuda.empty_cache()\n    evidence = Path(\'report/evidence\') / f\'{pred_key}.json\' if split == \'test\' else (\n        Path(\'work\') / f\'{pred_key}_val_score.json\')\n    cmd = [sys.executable, \'scripts/score.py\', \'--pred\', str(pred_path), \'--split\', split,\n           \'--coord\', cfg[\'coord\'], \'--name\', pred_key.upper(), \'--out\', str(evidence)]\n    if key != \'e0\':\n        run = json.loads((Path(\'work\') / key / \'final/run.json\').read_text(encoding=\'utf-8\'))\n        for k in (\'trainable_params\', \'peak_vram_gb\', \'train_minutes\'):\n            cmd.extend([\'--meta\', f\'{k}={run[k]}\'])\n    else:\n        cmd.extend([\'--meta\', \'trainable_params=0\', \'--meta\', \'train_minutes=0\'])\n    if resource_path.exists():\n        resources = json.loads(resource_path.read_text(encoding=\'utf-8\'))\n        for k, v in resources.items(): cmd.extend([\'--meta\', f\'{k}={v}\'])\n        if key == \'e0\':\n            cmd.extend([\'--meta\', f\'peak_vram_gb={resources["inference_peak_vram_gb"]}\'])\n    subprocess.run(cmd, check=True)\n    return json.loads(evidence.read_text(encoding=\'utf-8\'))\n\n\ndef run_ablation(configs, variable=\'data_size\'):\n    choices = {\'data_size\': [.1, 1.0], \'rank\': [4, 64], \'coord_scale\': [\'norm1000\', \'pixel\']}\n    if variable not in choices: raise ValueError(\'Choose data_size, rank or coord_scale\')\n    validation = {k: predict_and_score(c, k, \'val\')[\'micro\'][\'f1\'] for k, c in configs.items()}\n    best = max(validation, key=validation.get)\n    write_json(\'work/ablation_selection.json\', {\'selection_split\': \'val\', \'f1\': validation,\n        \'base_setup\': best, \'variable\': variable, \'values\': choices[variable]})\n    for value in choices[variable]:\n        cfg = copy.deepcopy(configs[best])\n        if variable == \'rank\': cfg[\'lora\'].update(r=value, alpha=2*value)\n        elif variable == \'data_size\': cfg[\'train_fraction\'] = value\n        else: cfg[\'coord\'] = value\n        tag = f\'e4_{variable}_{str(value).replace(".", "p")}\'\n        if cfg == configs[best]:\n            # The 100% or norm1000 reference can reuse the already-trained base.\n            predict_and_score(cfg, best, prediction_key=tag)\n        else:\n            train(cfg, tag)\n            predict_and_score(cfg, tag)\n    return best\n\n\ndef draw_examples(split=\'test\', prediction_key=\'e1\', count=6, coord=\'norm1000\'):\n    from PIL import Image, ImageDraw\n    from scripts.score import _gt_boxes, _match, parse_output\n    rows = read_jsonl(f\'data/splits/{split}.jsonl\')\n    preds = {r[\'image_id\']: r[\'raw\'] for r in read_jsonl(f\'work/{prediction_key}_{split}.jsonl\')}\n    ranked = []\n    for row in rows:\n        parsed = parse_output(preds[row[\'image_id\']])\n        norm = parsed or []\n        if coord == \'pixel\':\n            norm = [{\'label\': p[\'label\'], \'bbox\': tuple(v/s*1000 for v,s in zip(\n                p[\'bbox\'], (row[\'width\'], row[\'height\'], row[\'width\'], row[\'height\'])))} for p in norm]\n        per, _ = _match(_gt_boxes(row), norm)\n        error = sum(v[1]+v[2] for v in per.values())\n        ranked.append((error, row, norm, parsed is not None))\n    ranked.sort(key=lambda x: (x[0], x[1][\'image_id\']))\n    half = count//2\n    chosen = ranked[:half] + ranked[max(half, len(ranked)-(count-half)):]\n    directory = Path(\'report/figures\')\n    directory.mkdir(parents=True, exist_ok=True)\n    captions = [\'# Ground truth (green) and predictions (red)\', \'\']\n    for i, (error, row, pred, valid) in enumerate(chosen):\n        with Image.open(Path(\'data\') / row[\'image_path\']) as im: image = im.convert(\'RGB\')\n        draw = ImageDraw.Draw(image)\n        for boxes, color in ((_gt_boxes(row), \'lime\'), (pred, \'red\')):\n            for box in boxes:\n                xy = [v*s/1000 for v,s in zip(box[\'bbox\'],\n                    (row[\'width\'], row[\'height\'], row[\'width\'], row[\'height\']))]\n                draw.rectangle(xy, outline=color, width=2)\n                draw.text((xy[0], max(0, xy[1]-12)), box[\'label\'], fill=color)\n        file = f\'{prediction_key}_{i+1}.jpg\'\n        image.save(directory / file)\n        captions.extend([f\'![{row["image_id"]}]({file})\',\n            f\'{row["image_id"]}: FP + FN = {error}; valid JSON = {valid}.\', \'\'])\n    (directory / \'README.md\').write_text(\'\\n\'.join(captions), encoding=\'utf-8\')\n\n\ndef export_results():\n    evidence = sorted(Path(\'report/evidence\').glob(\'e*.json\'))\n    head = \'| Run | Trainable params | Peak VRAM GB | Minutes | F1 | CI95 | FP/image | Pothole F1 | Crack F1 | Manhole F1 |\'\n    lines = [head, \'|---|---|---|---|---|---|---|---|---|---|\']\n    for p in evidence:\n        d = json.loads(p.read_text(encoding=\'utf-8\'))\n        m = d[\'meta\']\n        cells = [p.stem, str(m.get(\'trainable_params\', 0)),\n                 f\'{m["peak_vram_gb"]:.2f}\' if \'peak_vram_gb\' in m else \'-\',\n                 f\'{m["train_minutes"]:.2f}\' if \'train_minutes\' in m else \'-\',\n                 f\'{d["micro"]["f1"]:.4f}\', f\'{d["f1_ci95"][0]:.4f} - {d["f1_ci95"][1]:.4f}\',\n                 f\'{d["fp_per_image"]:.3f}\'] + [f\'{d["per_class"][c]["f1"]:.4f}\' for c in CLASSES]\n        lines.append(\'| \' + \' | \'.join(cells) + \' |\')\n    Path(\'report/results.md\').write_text(\'\\n\'.join(lines)+\'\\n\', encoding=\'utf-8\')\n    meta = json.loads(Path(\'data/splits/meta.json\').read_text(encoding=\'utf-8\'))\n    report = Path(\'REPORT.md\')\n    text = report.read_text(encoding=\'utf-8\').replace(\'Họ tên: <điền họ tên>\',\n        f\'Họ tên: {STUDENT_NAME}\\nMSV: {STUDENT_ID}\\nMôi trường thực nghiệm: Kaggle T4\')\n    text = re.sub(r\'`dataset_version`:.*\', f\'`dataset_version`: {meta["dataset_version"]}\', text, count=1)\n    # Replace only the results table; measured answers still require personal analysis.\n    text = re.sub(r\'\\| Mã \\| Setup \\|.*?(?=\\n## 10)\', \'\\n\'.join(lines)+\'\\n\', text, flags=re.S)\n    report.write_text(text, encoding=\'utf-8\')\n    write_json(\'report/data_summary.json\', meta)\n    # Preserve small reproducibility logs in the submission, without any weights.\n    for p in Path(\'work\').glob(\'*/final/run.json\'):\n        write_json(Path(\'report/runs\') / (p.parent.parent.name + \'_run.json\'), json.loads(p.read_text(encoding=\'utf-8\')))\n    for p in Path(\'work\').glob(\'*/loss.json\'):\n        write_json(Path(\'report/runs\') / (p.parent.name + \'_loss.json\'), json.loads(p.read_text(encoding=\'utf-8\')))\n    for p in Path(\'work\').glob(\'*/overflow.json\'):\n        write_json(Path(\'report/runs\') / (p.parent.name + \'_overflow.json\'), json.loads(p.read_text(encoding=\'utf-8\')))\n    for p in Path(\'work\').glob(\'*_val_score.json\'):\n        write_json(Path(\'report/validation\') / p.name, json.loads(p.read_text(encoding=\'utf-8\')))\n    if Path(\'work/ablation_selection.json\').exists():\n        write_json(\'report/ablation_selection.json\', json.loads(Path(\'work/ablation_selection.json\').read_text(encoding=\'utf-8\')))\n    paths = [p for p in Path(\'report\').rglob(\'*\') if p.is_file()]\n    paths += [Path(\'REPORT.md\'), Path(\'kaggle_vlm.py\'), Path(\'lab_day10.ipynb\')]\n    paths += [Path(\'verify_student_code.py\'), Path(\'build_kaggle_notebook.py\')]\n    paths += [Path(\'restore_kaggle_output.py\')]\n    paths += list(Path(\'configs\').glob(\'*.yaml\'))\n    target = Path(\'submission_2A202602590.zip\')\n    with zipfile.ZipFile(target, \'w\', zipfile.ZIP_DEFLATED) as z:\n        for p in paths:\n            if p.exists(): z.write(p, str(p))\n    print(\'Download:\', target.resolve())\n    print(\'REPORT.md questions require analysis of your real outputs; no results are fabricated.\')\n    return target\n\n\ndef iou_reward(completion, gt_boxes):\n    """Optional E5 reward, following the official specification (no GRPO run)."""\n    from scripts.score import iou_xyxy, parse_output\n    pred = parse_output(completion)\n    if pred is None: return 0.0\n    if not gt_boxes and not pred: return 1.0\n    pairs = sorted(((iou_xyxy(g[\'bbox\'], p[\'bbox\']), i, j)\n        for i, g in enumerate(gt_boxes) for j, p in enumerate(pred)\n        if g[\'label\'] == p[\'label\']), reverse=True)\n    used_g, used_p, total = set(), set(), 0.\n    for iou, i, j in pairs:\n        if iou < .5: break\n        if i in used_g or j in used_p: continue\n        used_g.add(i)\n        used_p.add(j)\n        total += iou\n    return .2 + .8 * total / max(len(gt_boxes), len(pred), 1)\n\n\ndef run_all(variable=\'data_size\'):\n    predict_and_score(config_for(\'e0\'), \'e0\')\n    configs = {key: config_for(key) for key in (\'e1\', \'e2\', \'e3\')}\n    for key, cfg in configs.items():\n        train(cfg, key)\n        predict_and_score(cfg, key)\n    best = run_ablation(configs, variable)\n    draw_examples(prediction_key=best)\n    export_results()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--run-all\', action=\'store_true\')\n    parser.add_argument(\'--ablation\', choices=[\'data_size\', \'rank\', \'coord_scale\'], default=\'data_size\')\n    args = parser.parse_args()\n    if args.run_all: run_all(args.ablation)\n    else: parser.print_help()\n', encoding='utf-8')
Path('verify_student_code.py').write_text('"""CPU checks for student labels, coordinate conventions and optional reward."""\nimport json\nimport math\nimport re\nimport tempfile\nfrom pathlib import Path\nfrom kaggle_vlm import objects_to_answer, yolo_to_box, make_prompt, iou_reward\nfrom scripts.score import evaluate, _gt_boxes\n\n\ndef main():\n    entry = {\'image_id\': \'sample\', \'width\': 640, \'height\': 360,\n             \'objects\': [{\'label\': \'pothole\', \'yolo\': [.5, .5, .25, .2]}]}\n    norm = objects_to_answer(entry[\'objects\'])\n    pixel = objects_to_answer(entry[\'objects\'], coord=\'pixel\')\n    assert json.loads(norm)[0][\'bbox_2d\'] == [375, 400, 625, 600]\n    assert json.loads(pixel)[0][\'bbox_2d\'] == [240, 144, 400, 216]\n    assert objects_to_answer([]) == \'[]\'\n    assert yolo_to_box(\'crack\', [0, 0, .2, .2])[\'bbox_2d\'] == [0, 0, 100, 100]\n    for coord in (\'norm1000\', \'pixel\'):\n        for xc, yc in ((.5, .5), (0, 0), (1, 1), (.00001, .99999)):\n            box = yolo_to_box(\'crack\', [xc, yc, .000001, .000001], coord=coord)[\'bbox_2d\']\n            sx, sy = (1000, 1000) if coord == \'norm1000\' else (640, 360)\n            assert 0 <= box[0] < box[2] <= sx\n            assert 0 <= box[1] < box[3] <= sy\n        for w, h in ((0, .1), (.1, 0), (0, 0)):\n            assert yolo_to_box(\'crack\', [.5, .5, w, h], coord=coord) is None\n    assert objects_to_answer([{\'label\': \'crack\', \'yolo\': [.5, .5, 0, .1]}]) == \'[]\'\n    from kaggle_vlm import audit_annotations\n    audit = audit_annotations([{\'width\':640, \'height\':360, \'objects\':[\n        {\'label\': \'crack\', \'yolo\': [.5, .5, .000001, .000001]},\n        {\'label\': \'pothole\', \'yolo\': [.5, .5, 0, .1]}]}])\n    assert audit[\'expanded_tiny_boxes\'] == 1\n    assert audit[\'zero_area_labels_omitted_from_training_answers\'] == 1\n    for raw, scale in ((norm, \'norm1000\'), (pixel, \'pixel\')):\n        score = evaluate([entry], {\'sample\': raw}, coord=scale, n_boot=100)\n        assert score[\'micro\'][\'f1\'] == 1\n    assert evaluate([entry], {\'sample\': pixel}, n_boot=100)[\'micro\'][\'f1\'] == 0\n    objects = entry[\'objects\'] + [{\'label\': \'manhole\', \'yolo\': [.2, .2, .1, .1]}]\n    assert json.loads(objects_to_answer(objects))[0][\'label\'] == \'manhole\'\n    assert \'0-1000\' in make_prompt() and \'pixel\' in make_prompt(\'pixel\')\n    gt = _gt_boxes(entry)\n    assert math.isclose(iou_reward(norm, gt), 1)\n    assert iou_reward(\'invalid\', gt) == 0\n    assert iou_reward(\'[]\', []) == 1\n    assert iou_reward(\'[]\', gt) == .2\n    wrong = norm.replace(\'pothole\', \'crack\')\n    assert iou_reward(wrong, gt) == .2\n    duplicate = json.dumps(json.loads(norm)*2)\n    assert math.isclose(iou_reward(duplicate, gt), .6)\n    print(\'Student code: labels, scale roundtrip, ordering and 6 reward cases passed.\')\n    verify_tensors()\n    verify_quantization()\n    verify_amp_overflow()\n\n\ndef verify_amp_overflow():\n    import torch\n    from kaggle_vlm import amp_optimizer_step\n    parameter = torch.nn.Parameter(torch.tensor([2.]))\n    optimizer = torch.optim.SGD([parameter], lr=.1)\n    scaler = torch.amp.GradScaler(\'cpu\', init_scale=256.)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1.)\n    initial_epoch = scheduler.last_epoch\n    # Simulate a real AMP overflow after backward, then retry with the same loss.\n    for bad in (float(\'inf\'), float(\'nan\')):\n        original = parameter.detach().clone()\n        scale = scaler.get_scale()\n        scaler.scale(parameter.square().sum()).backward()\n        parameter.grad.fill_(bad)\n        updated, norm = amp_optimizer_step(optimizer, scaler, [parameter])\n        assert not updated and norm is None\n        assert torch.equal(parameter, original)\n        assert parameter.grad is None and scaler.get_scale() == scale*.5\n        assert scheduler.last_epoch == initial_epoch\n    scaler.scale(parameter.square().sum()).backward()\n    updated, norm = amp_optimizer_step(optimizer, scaler, [parameter])\n    assert updated and math.isclose(norm, 4.)\n    scheduler.step()\n    assert scheduler.last_epoch == initial_epoch+1\n    assert math.isclose(parameter.item(), 1.9, abs_tol=1e-6)\n    assert parameter.grad is None\n    # The fp32 aggregate norm can overflow even if every gradient is finite.\n    parameter = torch.nn.Parameter(torch.tensor([2., 2.]))\n    optimizer = torch.optim.SGD([parameter], lr=.1)\n    scaler = torch.amp.GradScaler(\'cpu\', init_scale=256.)\n    original = parameter.detach().clone()\n    scale = scaler.get_scale()\n    scaler.scale(parameter.square().sum()).backward()\n    parameter.grad.fill_(1e30)\n    updated, _ = amp_optimizer_step(optimizer, scaler, [parameter])\n    assert not updated and torch.equal(parameter, original)\n    assert parameter.grad is None and scaler.get_scale() == scale*.5\n    scaler.scale(parameter.square().sum()).backward()\n    assert amp_optimizer_step(optimizer, scaler, [parameter])[0]\n    print(\'Student code: AMP overflow skips bad updates, reduces scale and recovers on retry.\')\n\n\ndef verify_quantization():\n    import torch\n    from kaggle_vlm import QUANTIZATION_SKIP_MODULES, validate_quantization\n    # Match the skip rules in transformers 5.19 quantizers_utils.should_convert_module.\n    def converts(name, patterns):\n        return not any(re.match(f\'{key}\\\\.\', name) or re.match(key, name) or name.endswith(key)\n                       for key in patterns)\n    for name in (\'model.visual.merger.linear_fc1\',\n                 \'model.visual.deepstack_merger_list.0.linear_fc2\',\n                 \'model.visual.blocks.0.attn.qkv\', \'lm_head\'):\n        assert not converts(name, QUANTIZATION_SKIP_MODULES), name\n    assert converts(\'model.visual.merger.linear_fc1\', [\'visual\', \'lm_head\'])\n    assert converts(\'model.language_model.layers.0.self_attn.q_proj\', QUANTIZATION_SKIP_MODULES)\n    class Params4bit(torch.nn.Parameter): pass\n    class Model:\n        bad = False\n        def named_parameters(self):\n            yield \'model.visual.merger.linear_fc1.weight\', (\n                Params4bit(torch.ones(1)) if self.bad else torch.nn.Parameter(torch.ones(1)))\n            yield \'model.language_model.layers.0.self_attn.q_proj.weight\', Params4bit(torch.ones(1))\n            yield \'lm_head.weight\', torch.nn.Parameter(torch.ones(1))\n    model = Model()\n    validate_quantization(model)\n    model.bad = True\n    try: validate_quantization(model)\n    except RuntimeError: pass\n    else: raise AssertionError(\'Quantized vision weights must be rejected, even with a floating dtype\')\n    print(\'Student code: full-path quantization skips and early model validation passed.\')\n\n\ndef verify_tensors():\n    import torch\n    from PIL import Image\n    from kaggle_vlm import collate, encode_example\n    short = {\'input_ids\': torch.tensor([[1, 2]]), \'attention_mask\': torch.ones(1, 2, dtype=torch.long),\n        \'mm_token_type_ids\': torch.tensor([[0, 1]]), \'labels\': torch.tensor([[-100, 2]]),\n        \'pixel_values\': torch.ones(3, 4), \'image_grid_thw\': torch.tensor([[1, 2, 2]])}\n    long = {\'input_ids\': torch.tensor([[3, 4, 5]]), \'attention_mask\': torch.ones(1, 3, dtype=torch.long),\n        \'mm_token_type_ids\': torch.tensor([[0, 1, 0]]), \'labels\': torch.tensor([[-100, -100, 5]]),\n        \'pixel_values\': torch.ones(5, 4), \'image_grid_thw\': torch.tensor([[1, 2, 4]])}\n    batch = collate([short, long], pad_token_id=0)\n    assert batch[\'input_ids\'].tolist() == [[1, 2, 0], [3, 4, 5]]\n    assert batch[\'labels\'].tolist() == [[-100, 2, -100], [-100, -100, 5]]\n    assert batch[\'attention_mask\'].tolist() == [[1, 1, 0], [1, 1, 1]]\n    assert batch[\'mm_token_type_ids\'].tolist() == [[0, 1, 0], [0, 1, 0]]\n    assert batch[\'pixel_values\'].shape == (8, 4)\n    assert batch[\'image_grid_thw\'].shape == (2, 3)\n    class Processor:\n        bad_prefix = False\n        def apply_chat_template(self, messages, **kwargs):\n            ids = [1, 2, 3] if len(messages) == 1 else [1, 2, 3, 4, 5]\n            if len(messages) == 2 and self.bad_prefix: ids[0] = 99\n            return {\'input_ids\': torch.tensor([ids])}\n    Path(\'.pytest_cache\').mkdir(exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=\'.pytest_cache\') as tmp:\n        path = (Path(tmp) / \'image.jpg\').resolve()\n        Image.new(\'RGB\', (640, 360)).save(path)\n        row = {\'image_path\': str(path), \'width\': 640, \'height\': 360, \'objects\': []}\n        processor = Processor()\n        encoded = encode_example(processor, row)\n        assert encoded[\'labels\'].tolist() == [[-100, -100, -100, 4, 5]]\n        processor.bad_prefix = True\n        try: encode_example(processor, row)\n        except RuntimeError: pass\n        else: raise AssertionError(\'Inconsistent chat prefix must fail\')\n    print(\'Student code: variable-length padding, vision concatenation and loss masking passed.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
Path('restore_kaggle_output.py').write_text('"""Reuse completed experiments from previous Kaggle Notebook Outputs attached as Input.\n\nThe previous source module keeps its original cache fingerprint. Its core model,\nlabels and data functions must match the current implementation. Only the AMP\ntraining loop and result exporter are patched, without relabelling old evidence.\n"""\nimport ast\nimport importlib.util\nimport inspect\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nCORE_FUNCTIONS = (\n    \'yolo_to_box\', \'objects_to_answer\', \'audit_annotations\', \'make_prompt\',\n    \'user_message\', \'config_for\', \'fingerprint\', \'seed_everything\', \'load_model\',\n    \'validate_quantization\', \'apply_setup\', \'encode_example\', \'collate\',\n    \'selected_train_rows\', \'save_checkpoint\', \'restore_model\', \'predict_and_score\',\n)\n\n\ndef validate_prepared_data(directory):\n    """Check saved splits and all referenced images before restoring cached runs."""\n    from scripts.splitting import dataset_version\n    directory = Path(directory).resolve()\n    meta = json.loads((directory / \'splits/meta.json\').read_text(encoding=\'utf-8\'))\n    image_splits, seen_images, seen_sessions = [], set(), {}\n    for split in (\'train\', \'val\', \'test\'):\n        rows = [json.loads(line) for line in (directory / f\'splits/{split}.jsonl\')\n                .read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n        if not rows: raise RuntimeError(f\'Empty saved split: {split}\')\n        if len(rows) != meta[\'counts\'][split][\'images\']:\n            raise RuntimeError(f\'Saved split image count mismatch: {split}\')\n        for row in rows:\n            if row[\'image_id\'] in seen_images:\n                raise RuntimeError(\'Duplicate image in saved splits\')\n            seen_images.add(row[\'image_id\'])\n            session = row[\'session\']\n            if session in seen_sessions and seen_sessions[session] != split:\n                raise RuntimeError(\'Saved data has a session in multiple splits\')\n            seen_sessions[session] = split\n            image = (directory / row[\'image_path\']).resolve()\n            if not image.is_relative_to(directory) or not image.is_file():\n                raise RuntimeError(f\'Saved Output is missing an image: {row["image_path"]}\')\n            image_splits.append((row[\'image_id\'], split))\n    if dataset_version(image_splits) != meta[\'dataset_version\']:\n        raise RuntimeError(\'Saved dataset_version does not match image/split assignments\')\n    return meta\n\n\ndef prepare_or_restore_data(input_dir=\'/kaggle/input\', output_dir=\'data\'):\n    """Use the raw road dataset, or validated prepared data from prior Output."""\n    input_dir, output_dir = Path(input_dir), Path(output_dir)\n    if (output_dir / \'splits/meta.json\').is_file():\n        meta = validate_prepared_data(output_dir)\n        print(\'Reuse prepared data:\', output_dir.resolve(), meta[\'dataset_version\'])\n        return\n    raw = sorted({p.parent for p in input_dir.rglob(\'labels-YOLO\')\n        if p.is_dir() and (p.parent / \'images\').is_dir()})\n    if len(raw) > 1:\n        raise RuntimeError(\'Multiple raw road datasets found; attach one copy\')\n    if raw:\n        print(\'Raw dataset:\', raw[0])\n        subprocess.run([sys.executable, \'scripts/prepare_data.py\', \'--source\', str(raw[0]),\n                        \'--out\', str(output_dir)], check=True)\n        validate_prepared_data(output_dir)\n        return\n    prepared = sorted({p.parent.parent for p in input_dir.rglob(\'meta.json\')\n        if p.parent.name == \'splits\' and (p.parent.parent / \'images\').is_dir()})\n    valid, failures = [], []\n    for directory in prepared:\n        try: valid.append((directory, validate_prepared_data(directory)))\n        except (RuntimeError, KeyError, ValueError, FileNotFoundError) as error:\n            failures.append(f\'{directory}: {error}\')\n    if not valid:\n        raise RuntimeError(\'No raw dataset or complete prepared data found. Use Add Input > Datasets \'\n            \'to attach lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes. \'\n            + (\'Incomplete Output: \' + \'; \'.join(failures) if failures else \'\'))\n    # Multiple versions of the same Output can carry identical prepared splits.\n    signatures = {tuple(hashlib_file(directory / f\'splits/{name}\') for name in\n        (\'meta.json\', \'train.jsonl\', \'val.jsonl\', \'test.jsonl\')) for directory, _ in valid}\n    if len(signatures) > 1:\n        raise RuntimeError(\'Attached Outputs have different prepared data; attach the correct dataset\')\n    directory, meta = valid[0]\n    shutil.copytree(directory, output_dir, dirs_exist_ok=True)\n    print(\'Restored prepared data from Output:\', directory, meta[\'dataset_version\'])\n\n\ndef hashlib_file(path):\n    import hashlib\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef compatible_source(previous_source, current_source):\n    def functions(source):\n        return {node.name: ast.dump(node, include_attributes=False)\n                for node in ast.parse(source).body if isinstance(node, ast.FunctionDef)}\n    old, new = functions(previous_source), functions(current_source)\n    changed = [name for name in CORE_FUNCTIONS if old.get(name) != new.get(name)]\n    for constant in (\'MODEL_ID\', \'CLASSES\', \'QUANTIZATION_SKIP_MODULES\'):\n        def value(source):\n            for node in ast.parse(source).body:\n                if isinstance(node, ast.Assign) and any(\n                    isinstance(t, ast.Name) and t.id == constant for t in node.targets):\n                    return ast.literal_eval(node.value)\n            return None\n        if value(previous_source) != value(current_source): changed.append(constant)\n    if changed:\n        raise RuntimeError(\'Previous Output has different model/data settings: \' + \', \'.join(changed))\n\n\ndef extract_embedded_sources(notebook_path):\n    """Read literal module snapshots from code cells, without executing a notebook."""\n    notebook = json.loads(Path(notebook_path).read_text(encoding=\'utf-8\'))\n    for cell in notebook.get(\'cells\', []):\n        if cell.get(\'cell_type\') != \'code\': continue\n        source = cell.get(\'source\', [])\n        if isinstance(source, list): source = \'\'.join(source)\n        try: tree = ast.parse(source)\n        except SyntaxError: continue\n        for node in ast.walk(tree):\n            if not isinstance(node, ast.Call) or not node.args: continue\n            callee = node.func\n            if not isinstance(callee, ast.Attribute) or callee.attr != \'write_text\': continue\n            receiver = callee.value\n            if not isinstance(receiver, ast.Call) or not receiver.args: continue\n            filename, contents = receiver.args[0], node.args[0]\n            if not isinstance(filename, ast.Constant) or filename.value != \'kaggle_vlm.py\': continue\n            if isinstance(contents, ast.Constant) and isinstance(contents.value, str):\n                yield contents.value\n\n\ndef restore_from_output(current, input_dir=\'/kaggle/input\', output_root=None):\n    input_dir = Path(input_dir)\n    if not input_dir.exists(): return current\n    candidates = sorted({p.parent for p in input_dir.rglob(\'kaggle_vlm.py\')\n        if (p.parent / \'work/e1/final/run.json\').is_file()})\n    if not candidates:\n        print(\'No previous E1 Output attached; experiments run from scratch.\')\n        return current\n    destination = Path(output_root).resolve() if output_root else Path.cwd().resolve()\n    # Prefer the Output with the most completed experiments. Older attached Output\n    # can provide the source file used before in-memory AMP patches were applied.\n    def completed(root):\n        return len(list((root / \'work\').glob(\'*/final/run.json\')))\n    most = max(completed(p) for p in candidates)\n    best = [p for p in candidates if completed(p) == most]\n    if len(best) != 1:\n        raise RuntimeError(\'Multiple Outputs have the same completed run count; attach one latest Output\')\n    previous_root = best[0]\n    run = json.loads((previous_root / \'work/e1/final/run.json\').read_text(encoding=\'utf-8\'))\n    config = current.config_for(\'e1\')\n    if run.get(\'config\', config) != config:\n        raise RuntimeError(\'Previous E1 configuration differs from the current experiment\')\n    current_source = Path(current.__file__).read_text(encoding=\'utf-8\')\n    provenance_name = \'report/provenance/runtime_kaggle_vlm.py\'\n    source_paths = [previous_root / provenance_name, previous_root / \'kaggle_vlm.py\']\n    source_paths += [p / provenance_name for p in candidates] + [p / \'kaggle_vlm.py\' for p in candidates]\n    # Some Outputs retain the exact old source only inside the executed notebook.\n    # Extract literals safely; never rerun its setup, training or shell cells.\n    extracted = destination / \'work/source_candidates\'\n    for notebook_path in input_dir.rglob(\'*.ipynb\'):\n        try:\n            for source in extract_embedded_sources(notebook_path):\n                try: compatible_source(source, current_source)\n                except RuntimeError: continue\n                import hashlib\n                digest = hashlib.sha256(source.encode(\'utf-8\')).hexdigest()\n                path = extracted / (digest + \'.py\')\n                extracted.mkdir(parents=True, exist_ok=True)\n                if not path.exists(): path.write_text(source, encoding=\'utf-8\')\n                source_paths.append(path)\n        except (ValueError, SyntaxError, KeyError):\n            continue\n    previous = None\n    for previous_path in dict.fromkeys(source_paths):\n        if not previous_path.is_file(): continue\n        try:\n            compatible_source(previous_path.read_text(encoding=\'utf-8\'), current_source)\n        except RuntimeError:\n            continue\n        spec = importlib.util.spec_from_file_location(\'day10_previous_output\', previous_path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        if module.fingerprint(config) == run[\'fingerprint\']:\n            previous = module\n            break\n    if previous is None:\n        print(\'Notebook Output found:\', previous_root)\n        print(\'Expected E1 fingerprint:\', run[\'fingerprint\'])\n        raise RuntimeError(\'Cannot find the source file matching completed E1. Attach the original \'\n                           \'E1 Output as a second Input alongside the newest E2 Output.\')\n    # Verify every completed run before copying or reusing it.\n    for run_path in (previous_root / \'work\').glob(\'*/final/run.json\'):\n        saved = json.loads(run_path.read_text(encoding=\'utf-8\'))\n        if \'config\' in saved and saved[\'fingerprint\'] != previous.fingerprint(saved[\'config\']):\n            raise RuntimeError(f\'Incompatible checkpoint provenance: {run_path}\')\n    # Preserve every original manifest, source fingerprint and evidence value.\n    for folder in (\'work\', \'report\'):\n        src, dest = previous_root / folder, destination / folder\n        if src.is_dir():\n            for p in src.rglob(\'*\'):\n                if not p.is_file(): continue\n                target = dest / p.relative_to(src)\n                if target.exists(): continue\n                target.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copy2(p, target)\n    provenance = destination / provenance_name\n    provenance.parent.mkdir(parents=True, exist_ok=True)\n    if provenance.exists() and provenance.read_bytes() != previous_path.read_bytes():\n        raise RuntimeError(\'Working directory already has different checkpoint provenance\')\n    if not provenance.exists(): shutil.copy2(previous_path, provenance)\n    # Source remains read-only in Input, keeping old signatures reproducible.\n    for function in (\'amp_optimizer_step\', \'train\', \'export_results\'):\n        exec(inspect.getsource(getattr(current, function)), previous.__dict__)\n    cfg = previous.config_for(\'e1\')\n    run = json.loads((destination / \'work/e1/final/run.json\').read_text(encoding=\'utf-8\'))\n    if run[\'fingerprint\'] != previous.fingerprint(cfg):\n        raise RuntimeError(\'Restored E1 config or dataset version differs; cannot reuse its checkpoint\')\n    adapter = destination / \'work/e1/final/adapter_model.safetensors\'\n    if not adapter.is_file():\n        raise RuntimeError(\'Previous E1 Output is missing adapter_model.safetensors\')\n    print(\'Restored previous Output:\', previous_root)\n    reused = [p.parent.parent.name for p in (destination / \'work\').glob(\'*/final/run.json\')]\n    print(\'Reuse completed training:\', sorted(reused))\n    print(\'Unfinished experiments restart with AMP recovery; original fingerprints are preserved.\')\n    return previous\n', encoding='utf-8')
Path('build_kaggle_notebook.py').write_text('"""Embed student code in the notebook so a single .ipynb upload works on Kaggle."""\nimport json\nimport inspect\nfrom pathlib import Path\n\n\ndef main():\n    path = Path(\'lab_day10.ipynb\')\n    nb = json.loads(path.read_text(encoding=\'utf-8\'))\n    import kaggle_vlm as lab\n    hotfix_code = inspect.getsource(lab.amp_optimizer_step) + \'\\n\\n\' + inspect.getsource(lab.train)\n    hotfix = (\'# Paste into a cell in the existing Kaggle notebook, then rerun the training cell.\\n\'\n              \'# Keep the source file unchanged so completed E0/E1 cache signatures remain valid.\\n\'\n              \'import kaggle_vlm as lab\\n\\n\'\n              "code = r\'\'\'" + hotfix_code + "\'\'\'\\n"\n              "exec(code, lab.__dict__)\\n"\n              "print(\'AMP recovery patched in memory. Rerun the training cell; E2 restarts, E1 is reused.\')\\n")\n    Path(\'kaggle_amp_hotfix.py\').write_text(hotfix, encoding=\'utf-8\')\n    while len(nb[\'cells\']) < 23:\n        nb[\'cells\'].append({\'cell_type\': \'markdown\', \'metadata\': {}, \'source\': []})\n    def setcell(index, text):\n        nb[\'cells\'][index][\'source\'] = text.splitlines(True)\n        if nb[\'cells\'][index][\'cell_type\'] == \'code\':\n            nb[\'cells\'][index][\'outputs\'] = []\n            nb[\'cells\'][index][\'execution_count\'] = None\n\n    setcell(0, \'# Lab Day 10: Qwen3-VL-4B road damage grounding\\n\'\n        \'Họ tên: **Phạm Văn Kiên** · MSV: **2A202602590**\\n\\n\'\n        \'Kaggle T4: executable E0–E4 implementation. Run cells in order. \'\n        \'Metrics are generated only from real model predictions. E5 GRPO is optional \'\n        \'and is not executed; its IoU reward has CPU checks.\\n\')\n    setup = """from pathlib import Path\nimport os\nimport subprocess\nimport sys\n\nif Path(\'/kaggle/working\').is_dir():\n    project = Path(\'/kaggle/working/day10\')\n    if not (project / \'scripts/prepare_data.py\').is_file():\n        subprocess.run([\'git\', \'clone\',\n            \'https://github.com/pvksssss/K4-Track-4-Day10-VisionLanguageModel.git\',\n            str(project)], check=True)\n    os.chdir(project)\nelse:\n    project = Path.cwd()\n    if not (project / \'scripts/prepare_data.py\').is_file():\n        raise RuntimeError(\'Run inside the project folder or on Kaggle.\')\nprint(\'Project:\', project)\nsubprocess.run([\'nvidia-smi\', \'-L\'], check=True)\nsubprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'-r\',\n               \'requirements-colab.txt\', \'pytest\', \'matplotlib\'], check=True)\n# This lab uses bitsandbytes, not torchao. Kaggle\'s optional torchao 0.10.0\n# makes PEFT 0.21.2 fail when adding LoRA to ordinary fp16 vision Linear layers.\nif Path(\'/kaggle/working\').is_dir():\n    subprocess.run([sys.executable, \'-m\', \'pip\', \'uninstall\', \'-y\', \'torchao\'], check=True)\nimport importlib\nimportlib.invalidate_caches()\nfrom peft.import_utils import is_torchao_available\nis_torchao_available.cache_clear()\nif Path(\'/kaggle/working\').is_dir() and is_torchao_available():\n    raise RuntimeError(\'Unexpected torchao backend is still active; restart the kernel\')\n# Exercise ordinary Linear dispatch now, before hours of E0/E1/E2 execution.\nfrom peft import LoraConfig, get_peft_model\nimport torch\nprobe = get_peft_model(torch.nn.Sequential(torch.nn.Linear(2, 2, dtype=torch.float16)),\n    LoraConfig(r=1, lora_alpha=2, target_modules=[\'0\']))\nassert any(\'lora_A\' in name for name, _ in probe.named_parameters())\ndel probe\nprint(\'PEFT fp16 Linear adapter dispatch preflight passed.\')\n"""\n    setcell(2, setup)\n    module = Path(\'kaggle_vlm.py\').read_text(encoding=\'utf-8\')\n    checks = Path(\'verify_student_code.py\').read_text(encoding=\'utf-8\')\n    recovery = Path(\'restore_kaggle_output.py\').read_text(encoding=\'utf-8\')\n    builder = Path(__file__).read_text(encoding=\'utf-8\')\n    setcell(3, "STUDENT_NAME = \'Phạm Văn Kiên\'\\nSTUDENT_ID = \'2A202602590\'\\n"\n        "print(f\'{STUDENT_NAME} - {STUDENT_ID}\')\\n\\n"\n        "# Embedded implementation survives cloning the original assignment repo.\\n"\n        f"Path(\'kaggle_vlm.py\').write_text({module!r}, encoding=\'utf-8\')\\n"\n        f"Path(\'verify_student_code.py\').write_text({checks!r}, encoding=\'utf-8\')\\n"\n        f"Path(\'restore_kaggle_output.py\').write_text({recovery!r}, encoding=\'utf-8\')\\n"\n        f"Path(\'build_kaggle_notebook.py\').write_text({builder!r}, encoding=\'utf-8\')\\n"\n        "# Replace the cloned template with this implemented notebook for ZIP export.\\n"\n        "subprocess.run([sys.executable, \'build_kaggle_notebook.py\'], check=True)\\n"\n        "sys.path.insert(0, str(project))\\n"\n        "import importlib\\nimport kaggle_vlm as lab\\nimportlib.reload(lab)\\n"\n        "report_path = project / \'REPORT.md\'\\n"\n        "text = report_path.read_text(encoding=\'utf-8\')\\n"\n        "text = text.replace(\'Họ tên: <điền họ tên>\', "\n        "f\'Họ tên: {STUDENT_NAME}\\\\nMSV: {STUDENT_ID}\\\\nMôi trường thực nghiệm: Kaggle T4\')\\n"\n        "report_path.write_text(text, encoding=\'utf-8\')\\n")\n    setcell(1, \'## 0. Environment (Kaggle T4)\\n\'\n        \'Select GPU T4 (or T4 x2), enable Internet, and attach \'\n        \'lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes with Add Input. \'\n        \'The model uses GPU 0; two T4 cards do not pool VRAM.\\n\\n\'\n        \'To reuse completed E0/E1/E2 from the previous failed run, create a notebook from its \'\n        \'Output or attach that Output as Input. If it lacks prepared data/images \'\n        \'and data/splits, attach the original road dataset too. \'\n        \'The data cell restores completed experiments after \'\n        \'checking model/data compatibility. If it requests the original E1 \'\n        \'source Output, attach that earlier Output as well. Unfinished runs \'\n        \'start over with AMP recovery.\\n\')\n    setcell(4, \'## 1. Data: Kaggle Input and session splits\\n\'\n        \'Use the raw road dataset from Add Input, or reuse complete prepared data \'\n        \'(images and train/val/test splits) from the previous notebook Output.\\n\')\n    setcell(5, """from restore_kaggle_output import prepare_or_restore_data, restore_from_output\nprepare_or_restore_data()\nlab = restore_from_output(lab)\n""")\n    setcell(6, "from collections import Counter\\nfrom PIL import Image, ImageDraw\\n"\n        "import matplotlib.pyplot as plt\\n\\n"\n        "splits = {s: lab.read_jsonl(f\'data/splits/{s}.jsonl\') for s in (\'train\', \'val\', \'test\')}\\n"\n        "sessions = {s: {r[\'session\'] for r in rows} for s, rows in splits.items()}\\n"\n        "assert not sessions[\'train\'] & sessions[\'val\']\\n"\n        "assert not sessions[\'train\'] & sessions[\'test\']\\n"\n        "assert not sessions[\'val\'] & sessions[\'test\']\\n"\n        "for split, rows in splits.items():\\n"\n        "    print(split, len(rows), Counter(o[\'label\'] for r in rows for o in r[\'objects\']))\\n"\n        "    audit = lab.audit_annotations(rows)\\n"\n        "    print(\'Annotation audit:\', audit)\\n"\n        "    lab.write_json(f\'report/annotations/{split}.json\', audit)\\n"\n        "fig, axes = plt.subplots(2, 3, figsize=(15, 7))\\n"\n        "for ax, row in zip(axes.flat, splits[\'train\'][:6]):\\n"\n        "    with Image.open(Path(\'data\') / row[\'image_path\']) as im: image = im.convert(\'RGB\')\\n"\n        "    draw = ImageDraw.Draw(image)\\n"\n        "    boxes = __import__(\'json\').loads(lab.objects_to_answer(row[\'objects\'], coord=\'pixel\'))\\n"\n        "    for box in boxes:\\n"\n        "        draw.rectangle(box[\'bbox_2d\'], outline=\'lime\', width=2)\\n"\n        "        draw.text(tuple(box[\'bbox_2d\'][:2]), box[\'label\'], fill=\'lime\')\\n"\n        "    ax.imshow(image); ax.set_title(row[\'image_id\'], fontsize=8); ax.axis(\'off\')\\n"\n        "plt.tight_layout(); plt.show()\\n")\n    setcell(8, "# Implemented in the embedded kaggle_vlm module.\\n"\n        "yolo_to_box = lab.yolo_to_box\\nobjects_to_answer = lab.objects_to_answer\\n"\n        "print(objects_to_answer(splits[\'train\'][0][\'objects\']))\\n"\n        "subprocess.run([sys.executable, \'verify_student_code.py\'], check=True)\\n"\n        "subprocess.run([sys.executable, \'-m\', \'pytest\', \'-q\'], check=True)\\n")\n    setcell(10, "cfg0 = lab.config_for(\'e0\')\\n"\n        "e0 = lab.predict_and_score(cfg0, \'e0\')\\n"\n        "for row in lab.read_jsonl(\'work/e0_test.jsonl\')[:5]: print(row)\\n")\n    setcell(11, "print(\'E0 valid JSON rate:\', e0[\'parse_valid_rate\'])\\n"\n        "print(\'E0 micro metrics:\', e0[\'micro\'], \'CI95:\', e0[\'f1_ci95\'])\\n")\n    setcell(13, "from transformers import AutoProcessor\\n"\n        "import torch\\n"\n        "processor = AutoProcessor.from_pretrained(lab.MODEL_ID)\\n"\n        "processor.tokenizer.padding_side = \'right\'\\n"\n        "encoded = lab.encode_example(processor, splits[\'train\'][0])\\n"\n        "answer_tokens = encoded[\'input_ids\'][encoded[\'labels\'] != -100]\\n"\n        "print(\'Supervised tokens:\', processor.decode(answer_tokens, skip_special_tokens=False))\\n"\n        "image_token = processor.tokenizer.convert_tokens_to_ids(\'<|image_pad|>\')\\n"\n        "vision_tokens = int((encoded[\'input_ids\'] == image_token).sum())\\n"\n        "print(\'Vision token count:\', vision_tokens)\\n"\n        "assert vision_tokens == 220, \'Inspect image resolution and processor settings\'\\n"\n        "two = lab.collate([encoded, lab.encode_example(processor, splits[\'train\'][1])], "\n        "processor.tokenizer.pad_token_id)\\n"\n        "assert torch.all(two[\'labels\'][two[\'attention_mask\'] == 0] == -100)\\n"\n        "print({k: tuple(v.shape) for k, v in two.items()})\\n"\n        "del two, encoded, answer_tokens, processor\\n")\n    setcell(15, "# Consistent effective batch = 8 for all experiments.\\n"\n        "configs = {key: lab.config_for(key, batch_size=1, grad_accum=8) "\n        "for key in (\'e1\', \'e2\', \'e3\')}\\n"\n        "for key, cfg in configs.items(): print(key, cfg)\\n"\n        "# apply_setup validates 252 LLM / 348 LLM+ViT target modules during training.\\n"\n        "# AMP retries the same batch on overflow; only successful updates count as steps.\\n")\n    setcell(17, "runs = {}\\n"\n        "for key, cfg in configs.items():\\n"\n        "    runs[key] = lab.train(cfg, key)\\n"\n        "    # Score immediately so results survive interruptions between setups.\\n"\n        "    lab.predict_and_score(cfg, key)\\n")\n    setcell(19, "import json\\n"\n        "for key in (\'e0\', \'e1\', \'e2\', \'e3\'):\\n"\n        "    result = json.loads(Path(f\'report/evidence/{key}.json\').read_text())\\n"\n        "    print(key, result[\'micro\'], \'CI95:\', result[\'f1_ci95\'], \'resources:\', result[\'meta\'])\\n")\n    setcell(21, "# Choose exactly one variable. Default: 10% versus 100% training data.\\n"\n        "ABLATION_VARIABLE = \'data_size\'  # alternatives: \'rank\', \'coord_scale\'\\n"\n        "best_setup = lab.run_ablation(configs, ABLATION_VARIABLE)\\n"\n        "print(\'Selected using validation F1:\', best_setup)\\n"\n        "lab.draw_examples(prediction_key=best_setup, count=6)\\n"\n        "# Optional E5: IoU reward is implemented and tested; GRPO is not run here.\\n")\n    setcell(22, \'## Save and download results\\n\'\n        \'The next cell exports real metrics, six annotated images and resource/loss logs \'\n        \'into submission_2A202602590.zip. Complete the ten REPORT.md questions with \'\n        \'your own observations. Save the executed Kaggle notebook and download it separately \'\n        \'to preserve outputs. Checkpoints in work/ are excluded from the submission ZIP; \'\n        \'download them separately if needed. Interrupted inference continues at the next image; \'\n        \'interrupted training starts over, while completed training is reused.\\n\')\n    export_cell = {\'cell_type\': \'code\', \'metadata\': {}, \'execution_count\': None, \'outputs\': [],\n        \'source\': ["archive = lab.export_results()\\n",\n                   "print(Path(\'report/results.md\').read_text(encoding=\'utf-8\'))\\n",\n                   "from IPython.display import FileLink, display\\n",\n                   "display(FileLink(str(archive)))\\n"]}\n    nb[\'cells\'] = nb[\'cells\'][:23] + [export_cell]\n    path.write_text(json.dumps(nb, ensure_ascii=False, indent=1)+\'\\n\', encoding=\'utf-8\')\n    print(\'Generated self-contained Kaggle notebook:\', path)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
# Replace the cloned template with this implemented notebook for ZIP export.
subprocess.run([sys.executable, 'build_kaggle_notebook.py'], check=True)
sys.path.insert(0, str(project))
import importlib
import kaggle_vlm as lab
importlib.reload(lab)
report_path = project / 'REPORT.md'
text = report_path.read_text(encoding='utf-8')
text = text.replace('Họ tên: <điền họ tên>', f'Họ tên: {STUDENT_NAME}\nMSV: {STUDENT_ID}\nMôi trường thực nghiệm: Kaggle T4')
report_path.write_text(text, encoding='utf-8')


## 1. Data: Kaggle Input and session splits
Use the raw road dataset from Add Input, or reuse complete prepared data (images and train/val/test splits) from the previous notebook Output.


In [ ]:
from restore_kaggle_output import prepare_or_restore_data, restore_from_output
prepare_or_restore_data()
lab = restore_from_output(lab)


In [ ]:
from collections import Counter
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

splits = {s: lab.read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test')}
sessions = {s: {r['session'] for r in rows} for s, rows in splits.items()}
assert not sessions['train'] & sessions['val']
assert not sessions['train'] & sessions['test']
assert not sessions['val'] & sessions['test']
for split, rows in splits.items():
    print(split, len(rows), Counter(o['label'] for r in rows for o in r['objects']))
    audit = lab.audit_annotations(rows)
    print('Annotation audit:', audit)
    lab.write_json(f'report/annotations/{split}.json', audit)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, row in zip(axes.flat, splits['train'][:6]):
    with Image.open(Path('data') / row['image_path']) as im: image = im.convert('RGB')
    draw = ImageDraw.Draw(image)
    boxes = __import__('json').loads(lab.objects_to_answer(row['objects'], coord='pixel'))
    for box in boxes:
        draw.rectangle(box['bbox_2d'], outline='lime', width=2)
        draw.text(tuple(box['bbox_2d'][:2]), box['label'], fill='lime')
    ax.imshow(image); ax.set_title(row['image_id'], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Implemented in the embedded kaggle_vlm module.
yolo_to_box = lab.yolo_to_box
objects_to_answer = lab.objects_to_answer
print(objects_to_answer(splits['train'][0]['objects']))
subprocess.run([sys.executable, 'verify_student_code.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## 3. E0: zero-shot

In [ ]:
cfg0 = lab.config_for('e0')
e0 = lab.predict_and_score(cfg0, 'e0')
for row in lab.read_jsonl('work/e0_test.jsonl')[:5]: print(row)


In [ ]:
print('E0 valid JSON rate:', e0['parse_valid_rate'])
print('E0 micro metrics:', e0['micro'], 'CI95:', e0['f1_ci95'])


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
import torch
processor = AutoProcessor.from_pretrained(lab.MODEL_ID)
processor.tokenizer.padding_side = 'right'
encoded = lab.encode_example(processor, splits['train'][0])
answer_tokens = encoded['input_ids'][encoded['labels'] != -100]
print('Supervised tokens:', processor.decode(answer_tokens, skip_special_tokens=False))
image_token = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
vision_tokens = int((encoded['input_ids'] == image_token).sum())
print('Vision token count:', vision_tokens)
assert vision_tokens == 220, 'Inspect image resolution and processor settings'
two = lab.collate([encoded, lab.encode_example(processor, splits['train'][1])], processor.tokenizer.pad_token_id)
assert torch.all(two['labels'][two['attention_mask'] == 0] == -100)
print({k: tuple(v.shape) for k, v in two.items()})
del two, encoded, answer_tokens, processor


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Consistent effective batch = 8 for all experiments.
configs = {key: lab.config_for(key, batch_size=1, grad_accum=8) for key in ('e1', 'e2', 'e3')}
for key, cfg in configs.items(): print(key, cfg)
# apply_setup validates 252 LLM / 348 LLM+ViT target modules during training.
# AMP retries the same batch on overflow; only successful updates count as steps.


## 6. Vòng huấn luyện

In [ ]:
runs = {}
for key, cfg in configs.items():
    runs[key] = lab.train(cfg, key)
    # Score immediately so results survive interruptions between setups.
    lab.predict_and_score(cfg, key)


## 7. Suy luận và chấm E1–E3

In [ ]:
import json
for key in ('e0', 'e1', 'e2', 'e3'):
    result = json.loads(Path(f'report/evidence/{key}.json').read_text())
    print(key, result['micro'], 'CI95:', result['f1_ci95'], 'resources:', result['meta'])


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# Choose exactly one variable. Default: 10% versus 100% training data.
ABLATION_VARIABLE = 'data_size'  # alternatives: 'rank', 'coord_scale'
best_setup = lab.run_ablation(configs, ABLATION_VARIABLE)
print('Selected using validation F1:', best_setup)
lab.draw_examples(prediction_key=best_setup, count=6)
# Optional E5: IoU reward is implemented and tested; GRPO is not run here.


## Save and download results
The next cell exports real metrics, six annotated images and resource/loss logs into submission_2A202602590.zip. Complete the ten REPORT.md questions with your own observations. Save the executed Kaggle notebook and download it separately to preserve outputs. Checkpoints in work/ are excluded from the submission ZIP; download them separately if needed. Interrupted inference continues at the next image; interrupted training starts over, while completed training is reused.


In [ ]:
archive = lab.export_results()
print(Path('report/results.md').read_text(encoding='utf-8'))
from IPython.display import FileLink, display
display(FileLink(str(archive)))
